> **中文阅读版**：本文件是 `05b_feast_live.ipynb` 的 Markdown 翻译副本。代码、输出、图表和路径保持原样；专有名词和标识符保留英文。
# 使用 Feast 构建特征存储：实时集成

**第 26 章：MLOps 与治理**
**Docker 镜像**：`ml4t`
**书中参考**：第 26 章，第 26.6 节
**配套内容**：[`05_feast_feature_store`](05_feast_feature_store.ipynb)，它以 Polars 完成同样的工作

**学习目标**：
- 从磁盘上已有的 Parquet 文件搭建本地 Feast 仓库，在代码中声明实体、数据源和特征视图。
- 将应当回答的实体-时间戳对交给 Feast 以获取训练集，并像实时系统那样请求单个时刻的数据。
- 将 Feast 的答案与 `05` 中手写的连接逐特征核对，并把不一致视为配置缺陷，而非舍入差异。

`05` 手工构建离线连接、as-of 检索和偏斜度量，正是这一点让规则变得可见。本 notebook 将同样的任务交给开源特征存储 **Feast**，然后检查两者是否一致。

检查才是重点。特征存储是配置出来的，而不是写出来的：时间戳字段、实体键和存活时间都是声明，声明错误会产生一个能运行、能回答每个查询、却答错的存储。与一个规则可读的实现进行对比，正是发现这一点的方法。

**前置条件**：`pip install 'feast>=0.40'`（已包含在 `[mlops]` extra 中）。

In [1]:
"""Feature Store with Feast: the same retrievals as notebook 05, checked against it."""

'Feature Store with Feast: the same retrievals as notebook 05, checked against it.'

## 设置

`TRAINING_START`、`TRAINING_END` 和 `AS_OF_DATE` 限定两次检索的范围。若未设置，则像 `05` 中那样从案例研究的折几何结构推导，原因相同：在此处键入的日期是在断言某个折边界的位置，而重建案例研究会移动所有边界，字面量却不会移动。

`FEATURE_TTL_DAYS` 是两个 Feast 视图上声明的存活时间 (TTL)，值得仔细阅读。日频特征表上的两天意味着上一交易时段的值仍可服务，因此 Feast 对于没有自身行的日期查询，会通过向前延续上一行来回答。这正是存储的用途，也正是它使下面的一致性 (parity) 检查并非平凡：手写连接按精确键匹配，对此类日期返回空，因此两条路径只在事件集被限制为两个源实际都持有的键时才一致。

`SOURCE_PAD_DAYS` 在训练窗口两侧扩展交给 Feast 的 Parquet 副本，使 TTL 回看和 as-of 查询有行可触及。

`PARITY_TOLERANCE` 是两个实现逐特征允许的差异上限。它被设得远低于任何正确连接可能产生的差异，因为要找的是错误的行，而不是舍入误差：选错日期的存储会偏差该日特征变动的幅度，这比该容差大许多个数量级。

`N_SAMPLE_ASSETS`、`LIQUIDITY_WINDOW_DAYS` 和 `LIQUIDITY_RANK_DAYS` 根据训练窗口尾部的平均美元成交额，挑选 as-of 检索所询问的标的。

In [2]:
CASE_STUDY_ID = "us_equities_panel"
PRIMARY_LABEL = "fwd_ret_1d"
TRAINING_START = None
TRAINING_END = None
AS_OF_DATE = None
TRAINING_LOOKBACK_DAYS = 91
N_SAMPLE_ASSETS = 8
FEATURE_TTL_DAYS = 2
SOURCE_PAD_DAYS = 30
LIQUIDITY_WINDOW_DAYS = 21
LIQUIDITY_RANK_DAYS = 30
PARITY_TOLERANCE = 1e-10

In [3]:
import shutil
import tempfile
import warnings
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import yaml

from utils.cv_splits import generate_cv_splits
from utils.paths import get_case_study_dir

# Named, not blanket: a bare ignore would also hide the convergence and numerical
# warnings a reader needs to see.
warnings.filterwarnings("ignore", category=FutureWarning, module="feast")

CASE_DIR = get_case_study_dir(CASE_STUDY_ID)

FINANCIAL_FEATURES = ["past_ret_21d", "vol_21d", "rsi_14", "sharpe_21d"]
MODEL_FEATURES = ["garch_cond_vol", "ffd_log_price", "ffd_log_volume"]
ALL_FEATURES = FINANCIAL_FEATURES + MODEL_FEATURES

print("Feature Store with Feast: Live Integration")
print("=" * 60)

Feature Store with Feast: Live Integration


## 1. 导入 Feast

有四个类承载声明。`Entity` 命名行所描述的对象及其标识列。`FileSource` 指向一张表，并说明其哪一列是事件时间戳。`FeatureView` 将数据源绑定到带类型的特征列表和存活时间。`FeatureStore` 是这些声明注册进去的仓库，也是每个查询所经过的对象。

In [4]:
from feast import Entity, FeatureStore, FeatureView, Field, FileSource
from feast.data_format import ParquetFormat
from feast.types import Float64
from feast.value_type import ValueType

print("Feast imported successfully")

Feast imported successfully


## 2. 准备 Feast 兼容的源文件

Feast 的离线存储需要 datetime 时间戳来进行 Point-in-Time (PIT) 连接，而这些 Parquet 文件携带的是 `Date`。临时副本写入时会转换该列，且仅覆盖扩展后的窗口，因此这里不会写入案例研究自身的产物。在生产环境中，类型转换在物化数据源的作业中执行一次，而不是在每次查询旁边执行。

In [5]:
feast_tmp = tempfile.mkdtemp(prefix="feast_ml4t_")
feast_data_dir = Path(feast_tmp) / "data"
feast_data_dir.mkdir()

In [6]:
def folds_by_coverage(path, windows: list[tuple]) -> list[object]:
    """Pair each evaluation window with the artifact fold that covers it, by date.

    A fold id is a label the writer chose, not a property of the data, and two tools can
    number the same folds in opposite directions. Joining a stored id to a freshly
    generated one then pairs each date with the wrong fitted state and still succeeds,
    because both ids exist. Ordering both sides by the dates they cover is the pairing
    that holds whatever numbering wrote the file.
    """
    coverage = (
        pl.scan_parquet(path)
        .group_by("fold")
        .agg(pl.max("timestamp").alias("last_covered"))
        .collect()
        .sort("last_covered")
    )
    stored = coverage["fold"].to_list()
    if len(stored) != len(windows):
        raise ValueError(
            f"model_based.parquet carries {len(stored)} folds and this notebook derived "
            f"{len(windows)} evaluation windows. They cannot be paired by date, so the "
            "artifact does not describe the geometry this case study now declares; "
            "regenerate it against the current stage 04."
        )
    order = sorted(range(len(windows)), key=lambda i: windows[i])
    paired: list[object] = [None] * len(windows)
    for position, window_index in enumerate(order):
        paired[window_index] = stored[position]
    return paired


financial_src = CASE_DIR / "features" / "financial.parquet"
model_src = CASE_DIR / "features" / "model_based.parquet"

financial_feast_path = feast_data_dir / "financial.parquet"
model_feast_path = feast_data_dir / "model_based.parquet"

setup = yaml.safe_load((CASE_DIR / "config" / "setup.yaml").read_text())
holdout_start = pd.Timestamp(setup["evaluation"]["holdout_start"])
holdout_end = pd.Timestamp(setup["evaluation"]["holdout_end"])
timeline = pl.scan_parquet(financial_src).select("timestamp").unique().collect()
cv_splits = generate_cv_splits(timeline, case_study_id=CASE_STUDY_ID, label_buffer="1D")
# A feature value is servable for a session the model was evaluated on, so the date range
# handed to Feast is the union of the validation windows and the holdout.
validation_spans = [
    (pd.Timestamp(split["val_start"]).date(), pd.Timestamp(split["val_end"]).date())
    for split in cv_splits
]
model_windows = [*validation_spans, (holdout_start.date(), holdout_end.date())]
# Where the table carries a fold column each stock-date appears once per fold, so the
# window and the fold are applied together; where it does not, the date range selects it.
FOLD_KEYED_ARTIFACT = "fold" in pl.scan_parquet(model_src).collect_schema().names()
model_folds = (
    folds_by_coverage(model_src, model_windows)
    if FOLD_KEYED_ARTIFACT
    else [None] * len(model_windows)
)
# By date rather than by list position: which end of the list holds the latest window
# depends on the numbering convention, and the dates do not.
last_validation_span = max(validation_spans)

training_end = TRAINING_END or str(last_validation_span[1])
training_start = TRAINING_START or str(
    max(
        last_validation_span[0],
        pd.Timestamp(training_end).date() - pd.Timedelta(days=TRAINING_LOOKBACK_DAYS),
    )
)
as_of_date = AS_OF_DATE or str(
    pl.scan_parquet(financial_src)
    .filter(pl.col("timestamp") >= holdout_start.date())
    .select(pl.min("timestamp"))
    .collect()
    .item()
)
assert pd.Timestamp(training_end) < holdout_start, (
    f"training window ends {training_end}, on or after the holdout opens "
    f"{holdout_start.date()}; the offline join would mix data the model may not see"
)
print(f"Offline training window {training_start} to {training_end}; serving as of {as_of_date}")

train_start = pd.Timestamp(training_start).date()
train_end = pd.Timestamp(training_end).date()
# Padded on both sides so the lookback and the as-of query have rows to reach.
filter_start = train_start - timedelta(days=SOURCE_PAD_DAYS)
filter_end = train_end + timedelta(days=SOURCE_PAD_DAYS)

(
    pl.scan_parquet(financial_src)
    .select(["symbol", "timestamp", *FINANCIAL_FEATURES])
    .filter((pl.col("timestamp") >= filter_start) & (pl.col("timestamp") <= filter_end))
    .with_columns(pl.col("timestamp").cast(pl.Datetime("ns")))
    .collect()
    .write_parquet(financial_feast_path)
)

Offline training window 2015-09-30 to 2015-12-30; serving as of 2016-01-04


In [7]:
def window_terms(clipped: list[tuple]) -> list[pl.Expr]:
    """One predicate per evaluation window, to be OR-ed into a single filter.

    Where the artifact carries no fold column a window is a date range and nothing else.
    Where it does, the same window also names the fold whose fitted state that range is
    evaluated under, and the two are applied together: each fold holds its own value for
    the same stock-date, so a date range alone returns one row per fold with no rule for
    choosing between them.
    """
    if FOLD_KEYED_ARTIFACT:
        return [
            (pl.col("fold") == fold) & pl.col("timestamp").is_between(start, end)
            for fold, start, end in clipped
        ]
    return [pl.col("timestamp").is_between(start, end) for _, start, end in clipped]

面向模型的副本通过对评估窗口的并集施加一个过滤条件来构建，而不是为每个窗口生成一个帧再拼接：拼接会重复计算两个窗口覆盖的任意日期，而且会静默地这样做。

In [8]:
_span_lo = pd.Timestamp(filter_start).date()
_span_hi = pd.Timestamp(filter_end).date()
model_spans = [
    (fold, max(_span_lo, start), min(_span_hi, end))
    for fold, (start, end) in zip(model_folds, model_windows, strict=True)
    if start <= _span_hi and end >= _span_lo
]
if not model_spans:
    raise ValueError(
        f"No validation window and not the holdout covers {_span_lo}..{_span_hi}; "
        "the materialization range lies outside every window this model was evaluated on"
    )
model_features = (
    pl.scan_parquet(model_src)
    .filter(pl.any_horizontal(*window_terms(model_spans)))
    .select(["symbol", "timestamp", *MODEL_FEATURES])
    .collect()
    .sort(["timestamp", "symbol"])
)
assert not model_features.select(pl.struct("symbol", "timestamp").is_duplicated().any()).item(), (
    "a stock-date resolved to more than one model-based row; the evaluation windows "
    "this notebook derived are not disjoint over the materialization range"
)
model_features.with_columns(pl.col("timestamp").cast(pl.Datetime("ns"))).write_parquet(
    model_feast_path
)

fin_rows = pl.scan_parquet(financial_feast_path).select(pl.len()).collect().item()
mod_rows = pl.scan_parquet(model_feast_path).select(pl.len()).collect().item()
print(f"Financial features: {fin_rows:,} rows → {financial_feast_path.name}")
print(f"Model-based features: {mod_rows:,} rows → {model_feast_path.name}")

Financial features: 245,553 rows → financial.parquet
Model-based features: 243,239 rows → model_based.parquet


## 3. 创建 Feast 仓库

Feast 仓库需要一个 `feature_store.yaml`，声明项目名称、provider、registry 位置和存储后端。我们使用本地文件 provider，并为 registry 和在线存储都使用 SQLite。

In [9]:
feast_config = {
    "project": "ml4t_feature_store",
    "provider": "local",
    "registry": {"path": str(Path(feast_tmp) / "registry.db")},
    "online_store": {"type": "sqlite", "path": str(Path(feast_tmp) / "online.db")},
    "offline_store": {"type": "file"},
    "entity_key_serialization_version": 3,
}
config_path = Path(feast_tmp) / "feature_store.yaml"
config_path.write_text(yaml.dump(feast_config))
print(f"Feast repo: {feast_tmp}")
print(yaml.dump(feast_config, default_flow_style=False))

Feast repo: ~/scratch/feast_ml4t_3s_tmagq
entity_key_serialization_version: 3
offline_store:
  type: file
online_store:
  path: ~/scratch/feast_ml4t_3s_tmagq/online.db
  type: sqlite
project: ml4t_feature_store
provider: local
registry:
  path: ~/scratch/feast_ml4t_3s_tmagq/registry.db



## 4. 定义实体和特征视图

实体是 `symbol`，它是本仓库中每个案例研究产物使用的标识符。每个视图映射到一个 Parquet 数据源，并声明带类型的特征，因为 Feast 在注册时而非查询时验证模式。

存活时间限定了查询可以回溯多远。当存活时间为 `FEATURE_TTL_DAYS` 时，对于没有自身行的日期请求，会由该天数内最近的一行回答；早于该窗口的请求会返回空，而不是陈旧值。设得太短，合法查询会返回空；设得太长，存储会用不再描述所询问时刻的数据来回答。

In [10]:
symbol_entity = Entity(
    name="symbol",
    join_keys=["symbol"],
    value_type=ValueType.STRING,
    description="Stock ticker symbol",
)

financial_source = FileSource(
    path=str(financial_feast_path.resolve()),
    timestamp_field="timestamp",
    file_format=ParquetFormat(),
)

model_source = FileSource(
    path=str(model_feast_path.resolve()),
    timestamp_field="timestamp",
    file_format=ParquetFormat(),
)

financial_fv = FeatureView(
    name="financial_features",
    entities=[symbol_entity],
    schema=[Field(name=col, dtype=Float64) for col in FINANCIAL_FEATURES],
    source=financial_source,
    ttl=timedelta(days=FEATURE_TTL_DAYS),
)

model_fv = FeatureView(
    name="model_features",
    entities=[symbol_entity],
    schema=[Field(name=col, dtype=Float64) for col in MODEL_FEATURES],
    source=model_source,
    ttl=timedelta(days=FEATURE_TTL_DAYS),
)

print("Defined:")
print(f"  Entity:  {symbol_entity.name} (join_key={symbol_entity.join_key})")
print(f"  View 1:  {financial_fv.name} ({len(FINANCIAL_FEATURES)} features)")
print(f"  View 2:  {model_fv.name} ({len(MODEL_FEATURES)} features)")

Defined:
  Entity:  symbol (join_key=symbol)
  View 1:  financial_features (4 features)
  View 2:  model_features (3 features)


## 5. 应用特征存储

`store.apply()` 会在 Feast 注册表中注册实体和特征视图。
调用之后，特征存储知道去何处查找每个特征、按什么实体键进行连接，以及由哪个时间戳字段保证 Point-in-Time (PIT) 正确性。

In [11]:
store = FeatureStore(repo_path=feast_tmp)
store.apply([symbol_entity, financial_fv, model_fv])

registered_views = store.list_feature_views()
registered_entities = store.list_entities()
print(f"Registered {len(registered_entities)} entities, {len(registered_views)} feature views")
for fv in registered_views:
    print(f"  {fv.name}: {[f.name for f in fv.features]}")

Registered 1 entities, 2 feature views
  financial_features: ['past_ret_21d', 'vol_21d', 'rsi_14', 'sharpe_21d']
  model_features: ['garch_cond_vol', 'ffd_log_price', 'ffd_log_volume']


## 6. 离线检索：Point-in-Time (PIT) 连接

这是 Feast 围绕其构建的查询。它接收一个由 `(symbol, event_timestamp)` 对组成的数据框——即训练集需要得到答案的那些时刻——并对每一对返回该时刻或之前最近的、且在 time to live 内的特征值。这里的特征在收盘后已知，标签是下一交易时段的收益，而头寸不会早于那个下一交易时段才行动，因此这种配对对应下一根 bar 的决策。

### 限制事件集，以及为什么一致性检查需要这样做

对于没有自身特征行的日期，两种实现给出的答案不同。
Feast 会在其 time to live 内沿用上一行；`05` 中的精确键连接则不返回任何内容。两者就其自身而言都是正确的，因此在这样的日期上比较它们，会报告一个分歧，而该分歧属于问题本身的属性，而不属于任一实现的属性。

只要训练窗口跨越相邻验证窗口之间的 embargo 间隔，或延伸超过最后一个验证窗口，就会出现这样的日期。这个特定窗口是否包含任何此类日期，是折的几何结构（fold geometry）的事实，下面打印的计数说明了这一点：被丢弃的事件按缺失的是哪个来源进行拆分，因为模型特征中缺失的键正是该限制要处理的间隔，而金融特征中缺失的键则是另一个问题，只是也计入同一个总数。

In [12]:
requested = (
    pl.scan_parquet(CASE_DIR / "labels" / f"{PRIMARY_LABEL}.parquet")
    .filter(
        (pl.col("timestamp") >= pl.lit(pd.Timestamp(training_start).date()))
        & (pl.col("timestamp") <= pl.lit(pd.Timestamp(training_end).date()))
    )
    .select("symbol", "timestamp", pl.col(PRIMARY_LABEL).alias("label"))
    .collect()
)
financial_keys = (
    pl.scan_parquet(financial_feast_path)
    .select(pl.col("symbol"), pl.col("timestamp").cast(pl.Date))
    .unique()
    .collect()
)
labels = requested.join(
    model_features.select("symbol", "timestamp"), on=["symbol", "timestamp"]
).join(financial_keys, on=["symbol", "timestamp"])
if labels.is_empty():
    raise ValueError(
        f"No event in {training_start}..{training_end} has a row in both feature sources. "
        "Both join paths would return nothing, and the parity check below reads an "
        "empty difference as a match, so it would report every feature matching."
    )
if labels.height < requested.height:
    dropped = requested.join(
        labels.select("symbol", "timestamp"), on=["symbol", "timestamp"], how="anti"
    )
    no_model = dropped.join(
        model_features.select("symbol", "timestamp"), on=["symbol", "timestamp"], how="anti"
    ).height
    no_financial = dropped.join(financial_keys, on=["symbol", "timestamp"], how="anti").height
    # The two counts overlap where an event is missing from both sources.
    both = no_model + no_financial - dropped.height
    print(
        f"Dropped {dropped.height:,} of {requested.height:,} events: "
        f"{no_model:,} with no fitted model-feature vintage, "
        f"{no_financial:,} with no financial-feature row, "
        f"{both:,} missing from both."
    )

entity_df = labels.select(
    "symbol",
    pl.col("timestamp").cast(pl.Datetime("ns")).alias("event_timestamp"),
).to_pandas()

print(f"Entity DataFrame: {len(entity_df):,} rows")
print(f"Date range: {entity_df['event_timestamp'].min()} → {entity_df['event_timestamp'].max()}")
print(f"Symbols: {entity_df['symbol'].nunique()}")

Dropped 2 of 149,585 events: 0 with no fitted model-feature vintage, 2 with no financial-feature row, 0 missing from both.
Entity DataFrame: 149,583 rows
Date range: 2015-09-30 00:00:00 → 2015-12-30 00:00:00
Symbols: 2493


In [13]:
feature_refs = [f"financial_features:{col}" for col in FINANCIAL_FEATURES] + [
    f"model_features:{col}" for col in MODEL_FEATURES
]

feast_training = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs,
).to_df()

feast_training = feast_training.dropna(subset=ALL_FEATURES)
feast_training = feast_training.sort_values(["event_timestamp", "symbol"]).reset_index(drop=True)

print(f"Feast offline join: {len(feast_training):,} rows × {len(feast_training.columns)} columns")
feast_training.head()

Feast offline join: 149,583 rows × 9 columns


,symbol,event_timestamp,past_ret_21d,vol_21d,rsi_14,sharpe_21d,garch_cond_vol,ffd_log_price,ffd_log_volume
0,A,2015-09-30 00:00:00+00:00,-0.051786,0.294506,40.108077,-0.175840,0.187901,0.116286,0.623990
1,AAL,2015-09-30 00:00:00+00:00,-0.003848,0.334922,36.623519,-0.011490,0.528486,0.104252,1.169499
2,AAN,2015-09-30 00:00:00+00:00,-0.040302,0.283975,39.490029,-0.141921,0.187901,0.140993,0.494806
3,AAOI,2015-09-30 00:00:00+00:00,-0.088792,0.710573,46.907022,-0.124958,0.187901,0.222207,1.051132
4,AAON,2015-09-30 00:00:00+00:00,-0.062863,0.263935,38.021288,-0.238175,0.187901,0.100062,-0.228377


## 7. 将 Feast 与手写连接进行核对

同一事件集被回答两次：一次由 Feast 的 Point-in-Time (PIT) 连接回答，一次由 `05` 构建的精确键连接回答。两者应返回相同的行和相同的值。

这里出现分歧是配置缺陷，而不是数值问题。两条路径读取相同的 Parquet 文件，因此出现差异意味着其中一条选择了不同的行：错误的时间戳字段、不能唯一标识一行的实体键，或按 id 而非按日期选取的 fold vintage。上述每一种情况都会产生一个能运行但回答错误的存储，因此比较按特征逐个进行，并且容差设置得远低于任何正确连接可能产生的值。

此检查无法发现的一种错误配置是 time to live 设置得过长。这里比较的每个事件都在其自身日期上有一行特征，因此两条路径都取该行，不会发生回看；time to live 即使设为一年也不会改变任何一个值。要捕获这一点，需要事件自身没有对应行，且其事件年龄位于所声明限制的两侧，这是与此处不同的测试。

In [14]:
def polars_offline_join() -> pl.DataFrame:
    """Reproduce the manual offline join from notebook 05."""
    events = labels.select("symbol", "timestamp")

    financial = (
        pl.scan_parquet(financial_feast_path)
        .with_columns(pl.col("timestamp").cast(pl.Date))
        .select(["symbol", "timestamp", *FINANCIAL_FEATURES])
        .join(events.lazy(), on=["symbol", "timestamp"], how="inner")
    )
    model_based = (
        pl.scan_parquet(model_feast_path)
        .with_columns(pl.col("timestamp").cast(pl.Date))
        .select(["symbol", "timestamp", *MODEL_FEATURES])
        .join(events.lazy(), on=["symbol", "timestamp"], how="inner")
    )
    return (
        financial.join(model_based, on=["symbol", "timestamp"], how="inner")
        .drop_nulls(ALL_FEATURES)
        .collect()
        .sort(["timestamp", "symbol"])
    )


polars_result = polars_offline_join()
print(f"Polars offline join: {polars_result.height:,} rows")
print(f"Feast offline join:  {len(feast_training):,} rows")
print(f"Row count match:     {polars_result.height == len(feast_training)}")

Polars offline join: 149,583 rows
Feast offline join:  149,583 rows
Row count match:     True


In [15]:
# Compare the complete one-row-per-event results.
polars_pd = polars_result.to_pandas().rename(columns={"timestamp": "event_timestamp"})
# Align timestamp dtype/tz with Feast (ns + UTC) for the join key.
polars_pd["event_timestamp"] = pd.to_datetime(polars_pd["event_timestamp"], utc=True)
feast_training["event_timestamp"] = pd.to_datetime(feast_training["event_timestamp"], utc=True)

merged = polars_pd.merge(
    feast_training,
    on=["symbol", "event_timestamp"],
    how="inner",
    suffixes=("_polars", "_feast"),
)
assert len(merged) > 0, "Nothing to compare - a parity check over zero rows proves nothing."
assert len(polars_pd) == len(feast_training) == len(merged)

mismatches = []
for col in ALL_FEATURES:
    diff = (merged[f"{col}_polars"] - merged[f"{col}_feast"]).abs()
    max_diff = float(diff.max()) if len(diff) else 0.0
    mismatches.append(
        {"feature": col, "max_abs_diff": max_diff, "match": max_diff < PARITY_TOLERANCE}
    )

match_df = pd.DataFrame(mismatches)
match_df

,feature,max_abs_diff,match
0,past_ret_21d,0.0,True
1,vol_21d,0.0,True
2,rsi_14,0.0,True
3,sharpe_21d,0.0,True
4,garch_cond_vol,0.0,True
5,ffd_log_price,0.0,True
6,ffd_log_volume,0.0,True


In [16]:
n_match = int(match_df["match"].sum())
n_total = len(match_df)
mismatch_cols = match_df.loc[~match_df["match"], "feature"].tolist()
print(f"Parity: {n_match}/{n_total} features match within {PARITY_TOLERANCE:g}.")
if mismatch_cols:
    print(f"Differing columns: {mismatch_cols}")
assert n_match == n_total, f"Feast parity failed for {mismatch_cols}"

Parity: 7/7 features match within 1e-10.


上面的断言使这件事成为检查而非展示：只要有任何特征不一致，notebook 就会停止，因此看到下面单元格的读者就知道两条路径已经一致。

这种一致性所确立的内容比看起来更窄。数据源为每个 walk-forward 折保存一个拟合好的模型特征 vintage，两条路径在比较前都会选择在每个决策日期有效的 vintage。这一选择是两者都必须做对的部分；否则，一个股票-日期会解析到多行并使连接发生扇出，或者解析到用更晚数据拟合的 vintage。一致只说明两种实现做出了相同的选择，并不说明该选择是正确的——这正是上面 fold-by-date 配对的用途。

## 8. 在线式 as-of 检索

实时系统会调用 `get_online_features`，它读取由物化任务保持最新的键值存储。搭建这套系统需要本 notebook 并不具备的基础设施，因此这里用 `get_historical_features` 和每个实体一个时间戳来复现请求形态：同一个查询，但从离线存储得到答案。

这里重要的是形态——少量名称、一个时刻、每个名称一行——因为这是模型服务器每次决策时发出的形态。它没有复现的是延迟或一致性问题，即在线存储是否保存了离线存储本应回答的内容。

In [17]:
def sample_assets(n_assets: int) -> list[str]:
    from data import load_us_equities

    rank_from = pd.Timestamp(training_end).date() - pd.Timedelta(days=LIQUIDITY_RANK_DAYS)
    prices = load_us_equities(start_date=training_start, end_date=training_end)
    return (
        prices.lazy()
        .sort("symbol", "timestamp")
        .with_columns((pl.col("adj_close") * pl.col("adj_volume")).alias("dollar_volume"))
        .with_columns(
            pl.col("dollar_volume")
            .rolling_mean(LIQUIDITY_WINDOW_DAYS)
            .over("symbol")
            .alias("avg_dollar_volume")
        )
        .filter(
            (pl.col("timestamp") >= rank_from)
            & (pl.col("timestamp") <= pd.Timestamp(training_end).date())
        )
        .group_by("symbol")
        .agg(pl.col("avg_dollar_volume").mean().alias("mean_dollar_volume"))
        .sort("mean_dollar_volume", descending=True)
        .head(n_assets)
        .collect()
        .get_column("symbol")
        .to_list()
    )


sampled = sample_assets(N_SAMPLE_ASSETS)
as_of_entity_df = pd.DataFrame(
    {
        "symbol": sampled,
        "event_timestamp": pd.Timestamp(as_of_date),
    }
)

online_style = store.get_historical_features(
    entity_df=as_of_entity_df,
    features=feature_refs,
).to_df()

print(f"Online-style snapshot for {len(sampled)} assets as of {as_of_date}:")
online_style.sort_values("symbol")

Online-style snapshot for 8 assets as of 2016-01-04:


,symbol,event_timestamp,past_ret_21d,vol_21d,rsi_14,sharpe_21d,garch_cond_vol,ffd_log_price,ffd_log_volume
0,AAPL,2016-01-04 00:00:00+00:00,-0.093997,0.248240,34.324922,-0.378655,0.287989,0.155662,1.222692
2,AMZN,2016-01-04 00:00:00+00:00,-0.057721,0.322075,40.362438,-0.179216,0.339591,0.236471,1.476786
3,FB,2016-01-04 00:00:00+00:00,-0.036297,0.244539,42.246885,-0.148429,0.243191,0.208602,1.467711
1,GE,2016-01-04 00:00:00+00:00,0.032406,0.150843,54.797536,0.214832,0.164094,0.142864,0.989825
7,GOOG,2016-01-04 00:00:00+00:00,-0.026942,0.208622,46.880158,-0.129142,0.154066,0.387038,1.584194
6,GOOGL,2016-01-04 00:00:00+00:00,-0.023668,0.212159,46.769435,-0.111557,0.249444,0.255882,1.206573
5,MSFT,2016-01-04 00:00:00+00:00,-0.007426,0.236780,49.128828,-0.031363,0.242988,0.164954,1.286004
4,NFLX,2016-01-04 00:00:00+00:00,-0.147134,0.353459,36.591332,-0.416269,0.154066,0.175251,1.286394


## 9. 特征注册表内省

注册表是 Feast 自身对已声明内容的记录；把它读回来，就是运维人员在不打开声明它的代码的情况下回答“这个被服务的值来自哪里”的方式。它就是 `05` 手工组装的血缘表在该库中的版本。

In [18]:
registry_rows = []
for fv in store.list_feature_views():
    registry_rows.append(
        {
            "feature_view": fv.name,
            "entity": ", ".join(field.name for field in fv.entity_columns),
            "features": len(fv.features),
            "ttl": str(fv.ttl),
            "source_type": type(fv.batch_source).__name__,
        }
    )

registry_df = pd.DataFrame(registry_rows)
print("Feast Feature Registry:")
registry_df

Feast Feature Registry:


,feature_view,entity,features,ttl,source_type
0,financial_features,symbol,4,"2 days, 0:00:00",FileSource
1,model_features,symbol,3,"2 days, 0:00:00",FileSource


## 10. 清理

该仓库写在临时目录下，因此删除它不会留下任何东西。生产环境的注册表和在线存储会跨会话、跨读取它们的服务持续存在，这正是它们成为共享基础设施的主要原因。

In [19]:
shutil.rmtree(feast_tmp, ignore_errors=True)
print(f"Cleaned up temporary Feast repo: {feast_tmp}")

Cleaned up temporary Feast repo: ~/scratch/feast_ml4t_3s_tmagq


## 关键要点

1. 特征存储是配置出来的，而非直接编写。实体（entity）、时间戳字段和生存时间（TTL）都是声明；其中任何一项出错，都会产生一个能运行、能回答每个查询、但答案错误的存储。这正是检索之后设置检查的原因。
2. 将存储与一个规则可读的实现逐特征对比，并把容差设得远低于任何正确连接可能产生的偏差。要寻找的失败是错误行，其偏差等于该特征当天变动了多少，而不是舍入差异。
3. 在两者回答方式相同的问题上比较它们。存储会在其生存时间内将最后的值向前填充，而精确键连接不会，因此某个日期若没有自己的行，二者必然不一致；先限定事件集，并打印出被丢弃了什么以及原因。
4. 正确解读一致性，与获得一致性同样重要。一致性说明两条路径为每个决策日期选择了同一个拟合版本。它并不说明该选择是正确的，后者是另一项论证，由按日期配对的折来给出。
5. 这个库换来的是共享基础设施，而不是正确性。Feast 提供注册表、在线存储和多团队隔离，但需要先把它们搭建起来，并把其中的时间戳设为正确类型；`05` 不需要这些，而是直接在 Parquet 文件上做同样的检索。根据你需要其中哪些来做选择。

**已知限制**

- 这里的在线检索是 `get_historical_features`，每个实体一个时间戳。它复现的是请求形态，而不是在线存储，因此这里没有检验物化作业或两层之间的一致性，而这些正是线上特征存储自身故障所在。
- 注册表、在线存储和源副本都位于本 notebook 会删除的临时目录中。生产环境的注册表会持久化并共享，而它之所以具备治理属性，很大程度上是因为其他服务读取同一个注册表。
- 一致性检查覆盖一个案例研究上的一个训练窗口。它确立的是两个实现在这些事件上一致，而不是 Feast 声明对于具有不同折结构的窗口也正确。

**下一步**：参见 `06_mlflow_experiments` 了解使用 MLflow 进行实验跟踪。